In this notebook the model will be chosen and after defyining the threshold i will create full pipeline that works with the raw data

In [2]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix,
    precision_score, recall_score, f1_score, classification_report)

from sklearn.preprocessing import (OneHotEncoder, 
                                   StandardScaler, 
                                   OrdinalEncoder,
                                   PowerTransformer)
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin


from sklearn.base import clone
import lightgbm as lgb
from lightgbm import LGBMClassifier


from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    cross_val_predict
)

import pandas as pd
import numpy as np  
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib
import json

import sys

In [58]:
BASE_PATH= Path(os.getcwd()).parent

DATASET_PATH= BASE_PATH / 'dataset'

TRAIN_RAW_PATH= DATASET_PATH / 'split' / 'raw' / 'train_set.csv'
TEST_RAW_PATH= DATASET_PATH / 'split' / 'raw' / 'test_set.csv'
TRAIN_PREPRO_PATH= DATASET_PATH / 'split' /  'preprocessed' /'train_set.csv'
TEST_PREPRO_PATH= DATASET_PATH / 'split' /  'preprocessed' /'test_set.csv'


ARTIFACTS_PATH= BASE_PATH / 'artifacts'
FEATURES_PATH= ARTIFACTS_PATH / 'final_features.json'

MODEL_DATA_PATH=ARTIFACTS_PATH / 'model_data'
CUSTOM_MODEL_PATH=MODEL_DATA_PATH / 'models' / 'full_custom_final_model.joblib'
PRECUSTOM_MODEL_PATH=MODEL_DATA_PATH / 'models' / 'full_precustom_final_model.joblib'

THRESHOLD_PATH=MODEL_DATA_PATH/'thresholds.json'
THRESHOLD_PERFORMANCE_PATH=MODEL_DATA_PATH/'threshold_performance.json'


In [7]:
train_df=pd.read_csv(TRAIN_PREPRO_PATH)
test_df=pd.read_csv(TEST_PREPRO_PATH)

In [8]:
with open(FEATURES_PATH, "r", encoding='utf-8') as f:
    meta=json.load(f)

In [9]:
target=meta['target']
features=meta['final_features']
random_state=meta['random_state']
PAY_N=meta['PAY_N']
PAY_AMT=meta['PAY_AMT']
BILL_AMT=meta['BILL_AMT']

In [10]:
def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    ord_cols=PAY_N
    cat_cols = [c for c in features if c not in num_cols and c not in ord_cols]
    ordinal_categories = [[-2, -1, 0, 1, 2, 3]] * len(ord_cols)
    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True))
    ])
    ord_pipe= Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ordenc", OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ])
    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("ord", ord_pipe, ord_cols),
            ("cat", cat_pipe, cat_cols)
        ],
        remainder="drop"
    )
    return preprocessor


def make_pipeline(model, X: pd.DataFrame) -> Pipeline:
    preprocessor = make_preprocessor(X)
    return Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])


def evaluate_models_cv(
    X: pd.DataFrame,
    y: pd.Series,
    models: dict,
    n_splits: int = 5,
    seed: int = 42,
    return_oof: bool = True
) -> dict:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    results = {}
    for name, model in models.items():
        pipe = make_pipeline(model, X)

        # cross_validate gives per-fold scores + timing
        cv_out = cross_validate(
            pipe, X, y,
            cv=cv,
            scoring="roc_auc",
            return_train_score=False,
            n_jobs=-1
        )
        fold_scores = cv_out["test_score"]
        mean_auc = float(np.mean(fold_scores))
        std_auc = float(np.std(fold_scores))

        oof_auc = None
        if return_oof:
            # For AUC we need scores (probabilities or decision function)
            # cross_val_predict supports method='predict_proba' or 'decision_function'.
            # We try predict_proba first; if not available, fall back to decision_function.
            try:
                oof_scores = cross_val_predict(
                    pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1
                )[:, 1]
            except Exception:
                oof_scores = cross_val_predict(
                    pipe, X, y, cv=cv, method="decision_function", n_jobs=-1
                )
            oof_auc = float(roc_auc_score(y, oof_scores))

        results[name] = {
            "roc_auc_mean": mean_auc,
            "roc_auc_std": std_auc,
            "fold_scores": fold_scores,
            "oof_roc_auc": oof_auc,
            "fit_time_mean": float(np.mean(cv_out["fit_time"])),
            "score_time_mean": float(np.mean(cv_out["score_time"]))
        }

    return results


def print_cv_results(results: dict):
    rows = []
    for name, r in results.items():
        rows.append({
            "model": name,
            "roc_auc_mean": r["roc_auc_mean"],
            "roc_auc_std": r["roc_auc_std"],
            "oof_roc_auc": r["oof_roc_auc"],
            "fit_time_mean": r["fit_time_mean"]
        })
    summary = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
    print(summary.to_string(index=False))

    print("\nPer-fold scores:")
    for name, r in results.items():
        print(f"\n{name}: {np.round(r['fold_scores'], 4)}")

In [11]:
from xgboost import XGBClassifier

X=train_df.drop(columns=target, inplace=False)
y=train_df[target]

In [12]:
#training of different ml models with their default parameters
models = {
    "RandomForest": RandomForestClassifier(random_state=random_state),
    "GradientBoosting": GradientBoostingClassifier(random_state=random_state),
    "GaussianNB": GaussianNB(),
    "SVC_RBF": SVC(kernel="rbf", C=1.0, gamma="scale", random_state=random_state),
    "XGBClassifier": XGBClassifier(random_state=random_state, eval_metric="auc")
}

results = evaluate_models_cv(X, y, models=models, n_splits=5, seed=random_state, return_oof=True)
print_cv_results(results)

           model  roc_auc_mean  roc_auc_std  oof_roc_auc  fit_time_mean
GradientBoosting      0.780703     0.005989     0.780548       5.340006
   XGBClassifier      0.760814     0.005196     0.760807       0.672709
    RandomForest      0.760042     0.003088     0.760006       2.977248
      GaussianNB      0.738074     0.011513     0.738015       0.070904
         SVC_RBF      0.725136     0.008432     0.724943       9.228995

Per-fold scores:

RandomForest: [0.7595 0.7635 0.7561 0.7574 0.7637]

GradientBoosting: [0.784  0.7823 0.7688 0.7838 0.7846]

GaussianNB: [0.7536 0.7406 0.7191 0.7334 0.7437]

SVC_RBF: [0.7263 0.7267 0.7146 0.7393 0.7188]

XGBClassifier: [0.7634 0.7631 0.7514 0.7666 0.7596]


In [13]:
models={
    "LGBMClassifier": LGBMClassifier(random_state=random_state, verbose=-1),
}
results = evaluate_models_cv(X, y, models=models, n_splits=5, seed=random_state, return_oof=True)
print_cv_results(results)

         model  roc_auc_mean  roc_auc_std  oof_roc_auc  fit_time_mean
LGBMClassifier      0.779279     0.005427     0.779195       3.518722

Per-fold scores:

LGBMClassifier: [0.7814 0.7813 0.7685 0.7822 0.7831]


The best result can be seen from the GradientBoosting but still i am gonna choose the LGBMClassifier since it has much more tuning stuff that is more likely to increase the result

In [14]:
def sanity_check_shuffle_y(
    X: pd.DataFrame,
    y: pd.Series,
    model,
    seed: int = 42
) -> float:
    """
    Shuffle target; AUC should drop to ~0.50. If not, suspect leakage/bug.
    """
    rng = np.random.default_rng(seed)
    y_shuffled = pd.Series(rng.permutation(y.values), index=y.index)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    pipe = make_pipeline(model, X)

    scores = cross_validate(pipe, X, y_shuffled, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"]
    return float(np.mean(scores))

# Optional leakage sanity check on your best model:
leak_auc = sanity_check_shuffle_y(X, y, model=models["LGBMClassifier"], seed=42)
print("Shuffle-y sanity AUC (should be ~0.50):", leak_auc)

Shuffle-y sanity AUC (should be ~0.50): 0.4966397512514509


In [15]:
def make_objective(X: pd.DataFrame, y: pd.Series, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    def objective(trial: optuna.Trial) -> float:
        params = {
            "objective": "binary",
            "random_state": seed,
            "verbosity": -1,

            "n_estimators": 10000,

            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        }

        oof = np.zeros(len(y), dtype=float)

        for tr_idx, va_idx in cv.split(X, y):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            # Важно: строим pipeline на train-фолде (если у тебя dtype-логика внутри make_pipeline)
            model = LGBMClassifier(**params)
            pipe = make_pipeline(model, X_tr)  # твой pipeline (FE -> preprocess -> model)

            # 1) отдельно обучаем ВСЁ до модели и трансформим
            preproc = pipe[:-1]          # всё кроме последнего шага (модели)
            clf = pipe[-1]               # последняя модель (LGBMClassifier)

            X_tr_t = preproc.fit_transform(X_tr, y_tr)
            X_va_t = preproc.transform(X_va)

            # 2) обучаем модель на матрицах, и валидируем тоже на матрицах
            clf.fit(
                X_tr_t, y_tr,
                eval_set=[(X_va_t, y_va)],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(200, verbose=False)]
            )

            oof[va_idx] = clf.predict_proba(X_va_t)[:, 1]

        return roc_auc_score(y, oof)

    return objective

In [17]:
import warnings
messages=[
    "No further splits with positive gain, best gain: -inf",
    "X does not have valid feature names, but LGBMClassifier was fitted with feature names"
]
for msg in messages:
    warnings.filterwarnings(
        "ignore",
    message=msg
    )

In [18]:
# ---------- Run Optuna ----------
def tune_lgbm_optuna(X, y, n_trials=50, seed=random_state):
    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(make_objective(X, y, n_splits=5, seed=seed), n_trials=n_trials)

    print("Best AUC:", study.best_value)
    print("Best params:", study.best_params)
    return study

X = train_df.drop(columns=[target])
y = train_df[target]

study = tune_lgbm_optuna(X, y, n_trials=35, seed=42)

[I 2026-02-09 21:53:49,588] A new study created in memory with name: no-name-b832b10b-ded0-4638-af54-6d855894eee6
[I 2026-02-09 21:53:56,320] Trial 0 finished with value: 0.7775368483157888 and parameters: {'learning_rate': 0.02757359293934948, 'num_leaves': 230, 'min_child_samples': 152, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'reg_lambda': 0.000602521573620386}. Best is trial 0 with value: 0.7775368483157888.
[I 2026-02-09 21:54:08,931] Trial 1 finished with value: 0.7847559277604907 and parameters: {'learning_rate': 0.011703388679635262, 'num_leaves': 192, 'min_child_samples': 128, 'subsample': 0.9124217733388136, 'colsample_bytree': 0.7061753482887407, 'reg_lambda': 7.072114131472227}. Best is trial 1 with value: 0.7847559277604907.
[I 2026-02-09 21:54:13,848] Trial 2 finished with value: 0.7834414430503986 and parameters: {'learning_rate': 0.09528587217040241, 'num_leaves': 48, 'min_child_samples': 52, 'subsample': 0.7550213529560301, 'colsample_by

Best AUC: 0.7864658250956039
Best params: {'learning_rate': 0.010060697427449644, 'num_leaves': 43, 'min_child_samples': 121, 'subsample': 0.7267409340764938, 'colsample_bytree': 0.770882382469551, 'reg_lambda': 9.01339323745404}


In [19]:
best_params=study.best_params

In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

best_model = LGBMClassifier(
    **study.best_params,
    eval_metric="logloss",
    tree_method="hist",
    random_state=random_state,
    n_jobs=-1
)

pipe = make_pipeline(best_model, X)  # uses your make_pipeline + preprocessing

oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

print("OOF ROC AUC:", roc_auc_score(y, oof_proba))
print("OOF PR AUC (Average Precision):", average_precision_score(y, oof_proba))

OOF ROC AUC: 0.7807895472158117
OOF PR AUC (Average Precision): 0.5475829698477224


In [22]:
#saving the best hyperparameters
with open(MODEL_DATA_PATH / "best_params.json", "w", encoding="utf-8") as f:
    json.dump(study.best_params, f, indent=4)

In [23]:
def train_final_once(df: pd.DataFrame, target: str, best_params: dict, seed: int = 42):
    X = df.drop(columns=[target])
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    final_params = dict(best_params)
    final_params.update({
        "random_state": seed,
        "n_jobs": -1
    })

    model = LGBMClassifier(**final_params)
    pipe = make_pipeline(model, X_train)
    pipe.fit(X_train, y_train)

    test_scores = pipe.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_scores)

    print("FINAL TEST AUC:", test_auc)
    return pipe, test_auc

final_pipe, test_auc = train_final_once(train_df, target, best_params=best_params, seed=random_state)

FINAL TEST AUC: 0.7801339679315303


since the we can just guess what is more important finding the defaults or every class?. So i will create 2 threshold: for recall and f1

In [24]:
def pick_threshold_for_recall(y_true, proba_pos, target_recall=0.85):
    prec, rec, thr = precision_recall_curve(y_true, proba_pos)

    prec2, rec2 = prec[1:], rec[1:]

    mask = rec2 >= target_recall
    if not mask.any():
        idx = np.argmax(rec2)
        t = thr[idx]
        return t, prec2[idx], rec2[idx]

    idx = np.argmax(prec2[mask])
    t = thr[mask][idx]
    p = prec2[mask][idx]
    r = rec2[mask][idx]
    return t, p, r


# пример использования:
target_recall = 0.85
t, p, r = pick_threshold_for_recall(y, oof_proba,
                                    target_recall=target_recall)
print("Chosen threshold:", t)
print("Precision:", p)
print("Recall:", r)

Chosen threshold: 0.1631432155574505
Precision: 0.316968675375755
Recall: 0.8500659257864005


Chosing the threshold as recall and computing using the oof. I want the threshold to be around the 0.85

Note: since we are trying to balance and also maximize the recall the other metrics will drop such as precision for class 1 will drop

In [25]:
y_pred = (oof_proba >= t).astype(int)
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

print("TN FP FN TP:", tn, fp, fn, tp)
print("F1:", f1_score(y, y_pred))
print("Flag rate:", y_pred.mean())

TN FP FN TP: 8965 9726 796 4513
F1: 0.4617352158788623
Flag rate: 0.5932916666666667


In [30]:
pred = (oof_proba >= t).astype(int)

print("Confusion matrix:\n", confusion_matrix(y, pred))
print("Precision:", precision_score(y, pred))
print("Recall:", recall_score(y, pred))
print("F1:", f1_score(y, pred))
print("\nClassification report:\n", classification_report(y, pred))

Confusion matrix:
 [[8965 9726]
 [ 796 4513]]
Precision: 0.3169464147763186
Recall: 0.8500659257864005
F1: 0.4617352158788623

Classification report:
               precision    recall  f1-score   support

           0       0.92      0.48      0.63     18691
           1       0.32      0.85      0.46      5309

    accuracy                           0.56     24000
   macro avg       0.62      0.66      0.55     24000
weighted avg       0.79      0.56      0.59     24000



In [31]:
prec, rec, thresh = precision_recall_curve(y, oof_proba)

f1 = 2 * (prec[1:] * rec[1:]) / (prec[1:] + rec[1:] + 1e-12)

best_idx = np.argmax(f1)
best_threshold_f1 = thresh[best_idx]

print("Best threshold (max F1):", best_threshold_f1)
print("Precision:", prec[best_idx + 1], "Recall:", rec[best_idx + 1], "F1:", f1[best_idx])

Best threshold (max F1): 0.24893377185760632
Precision: 0.5147729191288198 Recall: 0.5743077792427953 F1: 0.5429131054126068


In [59]:
threshold ={
    'threshold_recall': float(t),
    'threshold_f1': float(best_threshold_f1)
}

In [60]:
with open(THRESHOLD_PATH, "w", encoding='utf-8') as f:
    json.dump(threshold, f, indent=2)

In [38]:
PROJECT_ROOT=Path(os.getcwd()).parent
sys.path.append(str(PROJECT_ROOT))

In [42]:
from src.feat_engineering import build_custom_pipeline

here we have the full ready pipeline with custom featureengineering. to check if the performance is the same with and without the custom featureengineering i would compare both of them where to one i will give the raw and for one the proccessed. 
>Note: raw and processed must be the same data

In [39]:
def check_similarity_finalpipe(pipe, train_df, target):
    X = train_df.drop(columns=[target])
    y = train_df[target]

    proba = pipe.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    print("Test AUC:", auc)

In [40]:
def check_similarity_custom_finalpipe(pipe, train_df, target):
    X = train_df.drop(columns=[target])
    y = train_df[target]

    pipe.fit(X, y)
    
    proba = pipe.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    print("Test AUC:", auc)

In [43]:
check_similarity_finalpipe(final_pipe, train_df, target)

Test AUC: 0.8071148353058599


In [44]:
train_raw_df=pd.read_csv(TRAIN_RAW_PATH)
custom_pipeline=build_custom_pipeline(best_params=best_params, random_state=random_state, final_pipe=final_pipe)
check_similarity_custom_finalpipe(custom_pipeline, train_raw_df, target)

Test AUC: 0.8083315980640997


The results should not be the same but similar. I mean like even if we just added the custom feature engineering class for the final_pipe the result may change due to other order of giving the data or etc.

result on test datasets

In [45]:
check_similarity_finalpipe(final_pipe, test_df, target)

Test AUC: 0.7720955783283243


In [46]:
test_raw_df=pd.read_csv(TEST_RAW_PATH)
proba=custom_pipeline.predict_proba(test_raw_df.drop(columns=[target]))[:, 1]
metric_on_test=roc_auc_score(test_raw_df[target], proba)
print("Final AUC on test set:", metric_on_test)

Final AUC on test set: 0.7755540131696605


In [49]:
def evaluate_fitted_pipeline(
    pipe,
    test_df: pd.DataFrame,
    target_col: str,
    threshold: float = 0.5,
) -> dict:
    
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col].to_numpy()

    proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = (proba >= threshold).astype(int)

    results = {
        "threshold": float(threshold),
        "roc_auc": float(roc_auc_score(y_test, proba)),
        "pr_auc": float(average_precision_score(y_test, proba)),
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),  # [[tn, fp],[fn,tp]]
        "n_test": int(len(y_test)),
        "pred_positive_rate": float(y_pred.mean()),
        "true_positive_rate": float(y_test.mean()),
    }
    return results

In [50]:
result_recall=evaluate_fitted_pipeline(custom_pipeline, test_raw_df, target_col=target, threshold=t)
result_f1=evaluate_fitted_pipeline(custom_pipeline, test_raw_df, target_col=target, threshold=best_threshold_f1)

In [51]:
print("Evaluation with recall-based threshold:")
print(result_recall)
print("\nEvaluation with F1-based threshold:")
print(result_f1)

Evaluation with recall-based threshold:
{'threshold': 0.1631432155574505, 'roc_auc': 0.7755540131696605, 'pr_auc': 0.5451895258308298, 'accuracy': 0.5628333333333333, 'precision': 0.3173618940248027, 'recall': 0.8485305199698568, 'f1': 0.46194871794871795, 'confusion_matrix': [[2251, 2422], [201, 1126]], 'n_test': 6000, 'pred_positive_rate': 0.5913333333333334, 'true_positive_rate': 0.22116666666666668}

Evaluation with F1-based threshold:
{'threshold': 0.24893377185760632, 'roc_auc': 0.7755540131696605, 'pr_auc': 0.5451895258308298, 'accuracy': 0.7833333333333333, 'precision': 0.5090786819098857, 'recall': 0.5704596834966089, 'f1': 0.5380241648898365, 'confusion_matrix': [[3943, 730], [570, 757]], 'n_test': 6000, 'pred_positive_rate': 0.24783333333333332, 'true_positive_rate': 0.22116666666666668}


In [52]:
performance_summary = {
    "recall_threshold": result_recall,
    "f1_threshold": result_f1
}

In [55]:
os.makedirs(THRESHOLD_PERFORMANCE_PATH.parent, exist_ok=True)
with open(THRESHOLD_PERFORMANCE_PATH, "w", encoding='utf-8') as f:
    json.dump(performance_summary, f, indent=2)

here the difference in performance based on the threshold can be clearly seen

In [56]:
os.makedirs(CUSTOM_MODEL_PATH.parent, exist_ok=True)
joblib.dump(custom_pipeline, CUSTOM_MODEL_PATH)

['c:\\Users\\User\\all_project\\projects_in_github\\taiwan2005_credict_card_project\\artifacts\\model_data\\models\\full_custom_final_model.joblib']

In [57]:
os.makedirs(PRECUSTOM_MODEL_PATH.parent, exist_ok=True)
joblib.dump(final_pipe, PRECUSTOM_MODEL_PATH)

['c:\\Users\\User\\all_project\\projects_in_github\\taiwan2005_credict_card_project\\artifacts\\model_data\\models\\full_precustom_final_model.joblib']